In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import altair as alt


prices_df = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/pp434/pp434_semi_anonymised_prices.parquet')
items_df = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/pp434/pp434_semi_anonymised_items.parquet')

In [2]:
# Show sample rows from prices_df and items_df
prices_df.sample(5)

,store_id,product_id,date,price,unit_price,loyalty_price,original_price
25049923,2,8089442,2023-08-15,8.00,80p / 75cl,NaN,9.5
25283730,3,1000383164418,2024-04-29,3.50,£3.50/75c3,NaN,NaN
14795749,3,1000157579174,2024-12-03,7.40,£2.20/100g,NaN,NaN
26195832,2,6691085,2023-10-03,2.00,£4.00 / ltr,NaN,2.0
17529046,3,1000200155857,2025-02-04,2.82,£4.41/kg,NaN,NaN


In [3]:
items_df.sample(5)

,store_id,product_id,segment_code,description
50931,3,910003006401,CP0111303,"BREAD ROLLS, BUNS, BAGUETTES AND OTHER LOAVES"
39145,2,8169419,CP0117903,"CRISPS, POTATO"
22049,5,4061462078545,CP0119101,"PRE-COOKED DISHES BASED ON MEAT, FISH, VEGETAB..."
40572,3,1000383211184,CP0118601,ICE CREAM TUBS (INCL. YOGHURT)
53932,2,8132392,CP0117903,"CRISPS, POTATO"


In [4]:
# Merge prices and items on store_id and product_id
df = pd.merge(prices_df, items_df, on=['store_id', 'product_id'], how='inner')
df.sample(5)

,store_id,product_id,date,price,unit_price,loyalty_price,original_price,segment_code,description
2833472,1,312579385,2025-05-16,4.80,10.67 / kg,NaN,NaN,CP0119101,"PRE-COOKED DISHES BASED ON MEAT, FISH, VEGETAB..."
3267578,4,609733011,2024-04-08,2.49,0.711 per 100g,NaN,NaN,CP0119304,PRE-MADE SAUCES (E.G. BOLOGNESE)
1779414,2,1117011,2024-07-23,1.79,£7.16 / kg,NaN,1.79,CP0115201,"BUTTER, DERIVED FROM MILK"
1162539,5,4088600547510,2024-09-11,1.79,£4.54 / kg,NaN,1.79,CP0112501,SAUSAGES AND SIMILAR MEAT PRODUCTS
393698,1,315071548,2024-12-14,3.15,1.85 / 100g,2.5,NaN,CP0111310,"BISCUITS, SAVOURY"


In [5]:
df.description.unique()

array(['RICE, IN ALL FORMS (EXCL. RICE FLOUR)', 'FLOUR, WHEAT-BASED',
       'BREAD, WHITE', 'BREAD, BROWN OR SEEDED',
       'BREAD ROLLS, BUNS, BAGUETTES AND OTHER LOAVES',
       'FLATBREADS, THINS AND PITTAS',
       'BREAD SIDE DISHES (E.G. GARLIC BREAD)',
       'OTHER BREAKFAST BAKERY PRODUCTS', 'BISCUITS, SWEET',
       'BISCUITS, SAVOURY', 'CAKES, TARTS AND SWEET PIES',
       'BREAKFAST CEREALS', 'CEREAL BARS AND CEREAL-BASED SNACKS',
       'OATS AND PORRIDGE', 'PASTA AND NOODLES, DRY OR FRESH',
       'PASTA AND NOODLES, PACKET OR POT', 'COUSCOUS',
       'MEAT OF COWS, FRESH, CHILLED OR FROZEN',
       'MEAT OF PIGS, FRESH, CHILLED OR FROZEN',
       'MEAT OF GOATS, LAMBS AND SHEEP, FRESH, CHILLED OR FROZEN',
       'MEAT OF CHICKEN, FRESH, CHILLED OR FROZEN',
       'COOKED HAM AND CONTINENTAL MEATS (E.G. SALAMI)',
       'COOKED POULTRY, SLICES AND DELI FOODS',
       'PORK, DRIED, SALTED OR SMOKED',
       'SAUSAGES AND SIMILAR MEAT PRODUCTS',
       'BREADED CHICKEN AN

In [25]:
_coffee_df = df.query("description == 'COFFEE'")

# This gives us a df with every price observation for frozen fruit products
# But we want the mean price over time

_coffee_avg_df = _coffee_df.groupby(['date']).agg({'price': 'mean'}).reset_index()


alt.Chart(_coffee_avg_df).mark_line(
    interpolate='monotone',
).encode(
    x=alt.X('date:T', title=''),
    y=alt.Y('price:Q', title='Mean price'),
).properties(
    title='Average Price of Coffee',
)

alt.Chart(...)

In [7]:
items_df.query("description == 'COFFEE'")

,store_id,product_id,segment_code,description
33,1,311667052,CP0122001,COFFEE
41,1,315054460,CP0122001,COFFEE
76,4,110832861,CP0122001,COFFEE
106,2,7801041,CP0122001,COFFEE
141,2,7849317,CP0122001,COFFEE
...,...,...,...,...
80031,3,1000383188824,CP0122001,COFFEE
80107,2,7983563,CP0122001,COFFEE
80175,1,315147051,CP0122001,COFFEE
80201,3,7568621,CP0122001,COFFEE


In [8]:
# Calculate median and mean prices for each store
store_prices = df.copy()
median_prices = store_prices.groupby(['store_id']).agg({'price': ['median', 'mean']}).reset_index()
median_prices.columns = ['store_id', 'median_price', 'mean_price']
median_prices

,store_id,median_price,mean_price
0,1,2.50,4.169827
1,2,2.55,4.205874
2,3,2.25,3.713805
3,4,2.25,3.559485
4,5,1.79,2.598145
5,8,2.50,3.919917
6,9,2.50,3.282282


In [9]:
# Prepare data for grouped bar chart
median_prices_melted = median_prices.melt(id_vars='store_id', value_vars=['median_price', 'mean_price'], var_name='price_type', value_name='price')
median_prices_melted['store_id'] = "Store " + median_prices_melted['store_id'].astype(str)

# Plot grouped bar chart
alt.Chart(median_prices_melted).mark_bar().encode(
    column=alt.Column('store_id', title=''),
    x=alt.X('price_type', title='', axis=alt.Axis(labels=False)),
    y=alt.Y('price', title='', axis={"labelExpr": "'£' + datum.label", "labelOverlap": False}),
    color='price_type'
).properties(
    title = {
        'text': "Prices by store",
        'subtitle': ["Mean and median prices", ""]
    },
    width=100
)

alt.Chart(...)

In [10]:
df.price.describe()

count    4.491682e+06
mean     3.837540e+00
std      4.944335e+00
min      1.800000e-01
25%      1.500000e+00
50%      2.490000e+00
75%      3.900000e+00
max      6.700000e+01
Name: price, dtype: float64

In [12]:
# Create a copy of the original DataFrame
hist_df = prices_df.copy()

# Round the 'price' column to 1 decimal place to group prices into rounded intervals
hist_df['rounded_price'] = hist_df['price'].round(1)

# Group by the rounded prices and count the occurrences of each rounded price
hist_df = hist_df.groupby('rounded_price').agg({'price': 'count'}).reset_index()

# Filter out rows where the rounded price is greater than 10
hist_df = hist_df.query("rounded_price <= 10")

# Rename the columns for clarity: 'rounded_price' to 'price', and the count to 'density'
hist_df.columns = ['price', 'density']

# Normalize the density values to calculate the relative frequency (density)
hist_df['density'] = hist_df['density'] / hist_df['density'].sum()

# Create a histogram using Altair
histogram = alt.Chart(hist_df).mark_bar(
    width=5
).encode(
    x=alt.X('price:Q',  title='', axis={"labelExpr": "'£'+datum.value"}),  # Bin the 'price' values into 20 bins
    y=alt.Y('density:Q', title='Density'),  # Plot the normalized density on the y-axis,
    tooltip=['price', 'density']  # Show the 'price' and 'density' values on hover
)

# Display the histogram
histogram

alt.Chart(...)

In [19]:
prices_df.columns

Index(['store_id', 'product_id', 'date', 'price', 'unit_price',
       'loyalty_price', 'original_price'],
      dtype='object')

In [20]:
bread_items = items_df[items_df['description'].str.contains('bread', case=False)]
bread_items

,store_id,product_id,segment_code,description
9,5,4088600308593,CP0112502,BREADED CHICKEN AND SIMILAR POULTRY PREPARATIONS
17,9,48103,CP0113301,"BREADED AND BATTERED FISH, CHILLED OR FROZEN"
54,4,607357011,CP0111304,"FLATBREADS, THINS AND PITTAS"
57,3,5732573,CP0112502,BREADED CHICKEN AND SIMILAR POULTRY PREPARATIONS
82,5,4061463957306,CP0112502,BREADED CHICKEN AND SIMILAR POULTRY PREPARATIONS
...,...,...,...,...
80227,4,268685011,CP0111304,"FLATBREADS, THINS AND PITTAS"
80237,1,321783820,CP0111303,"BREAD ROLLS, BUNS, BAGUETTES AND OTHER LOAVES"
80248,9,10706,CP0111301,"BREAD, WHITE"
80276,3,9137607,CP0111303,"BREAD ROLLS, BUNS, BAGUETTES AND OTHER LOAVES"
